# Tutorial 7: Motor Control and Timing

So far we've focused on cognition—retrieving memories, making decisions, learning. But in the real world, cognition leads to action. ACT-R includes a motor module that simulates physical movements like typing and clicking, complete with timing predictions based on Fitts's Law. This lets us model not just what people think, but how long it takes them to actually execute their actions.

## 1. Introduction to Motor Control

ACT-R includes a motor module that simulates physical actions like typing, clicking, and moving a mouse. This allows models to interact with interfaces in a way that approximates human timing. The motor module handles both preparation time (getting ready to act) and execution time (the physical movement itself).

In [ ]:
import pyactr as actr
import numpy as np

model = actr.ACTRModel()

actr.chunktype("type_goal", "state letter")

model.productionstring(
    name="press_key",
    string='''
    =g>
        isa type_goal
        state ready
        letter =key
    ?manual>
        state free
    ==>
    =g>
        state pressing
    +manual>
        isa _manual
        cmd press_key
        key =key
'''
)

print("Motor model created!")
print("Key press takes ~250ms (preparation + execution)")

## 2. Typing Model with Realistic Timing

Typing involves more than just pressing keys—you need to retrieve what to type next and coordinate finger movements. Here's a model that types a word, including the time to retrieve each letter from memory.

In [ ]:
import pyactr as actr

typing_model = actr.ACTRModel()
typing_model.model_parameters["motor_prepared"] = True

actr.chunktype("typing_goal", "state word current_pos")
actr.chunktype("letter_sequence", "position letter")

word = "cat"
for i, letter in enumerate(word):
    typing_model.decmem.add(
        actr.makechunk(
            chunktype="letter_sequence",
            position=str(i),
            letter=letter
        )
    )

typing_model.productionstring(
    name="get_letter",
    string='''
    =g>
        isa typing_goal
        state typing
        current_pos =pos
    ?manual>
        state free
    ==>
    =g>
        state retrieving
    +retrieval>
        isa letter_sequence
        position =pos
'''
)

typing_model.productionstring(
    name="type_letter",
    string='''
    =g>
        isa typing_goal
        state retrieving
        current_pos =pos
    =retrieval>
        isa letter_sequence
        letter =letter
    ?manual>
        state free
    ==>
    =g>
        state typing
        current_pos =pos
    +manual>
        isa _manual
        cmd press_key
        key =letter
    ~retrieval>
'''
)

print("Typing model ready!")
print(f"Will type: {word}")

## 3. Mouse Movement and Fitts's Law

ACT-R implements Fitts's Law for mouse movements: movement time increases logarithmically with distance and decreases with target size. The formula is time = a + b × log₂(2D/W), where D is distance and W is target width.

In [ ]:
import pyactr as actr
import math

mouse_model = actr.ACTRModel()

actr.chunktype("click_goal", "state target_x target_y")

mouse_model.productionstring(
    name="move_mouse",
    string='''
    =g>
        isa click_goal
        state ready
        target_x =x
        target_y =y
    ?manual>
        state free
    ==>
    =g>
        state moving
    +manual>
        isa _manual
        cmd move_cursor
        screen_x =x
        screen_y =y
'''
)

mouse_model.productionstring(
    name="click_mouse",
    string='''
    =g>
        isa click_goal
        state moving
    ?manual>
        state free
    ==>
    =g>
        state done
    +manual>
        isa _manual
        cmd click_mouse
'''
)

def fitts_time(distance, width, a=0.1, b=0.1):
    """Calculate movement time using Fitts's Law"""
    return a + b * math.log2(2 * distance / width)

distance = 200
target_width = 40
time = fitts_time(distance, target_width)

print(f"Fitts's Law prediction: {time:.3f} seconds")
print("ACT-R applies this automatically for mouse movements")

## 4. Dual-Task Performance: Thinking While Acting

One of the interesting features of ACT-R is that it can model parallel processing. Motor actions don't block cognition—you can retrieve from memory or make decisions while your hand is moving. This mirrors real human multitasking.

In [ ]:
import pyactr as actr

dual_model = actr.ACTRModel()

actr.chunktype("dual_goal", "typing_state count motor_busy")
actr.chunktype("count_fact", "current next")

for i in range(1, 5):
    dual_model.decmem.add(
        actr.makechunk(
            chunktype="count_fact",
            current=str(i),
            next=str(i+1)
        )
    )

dual_model.productionstring(
    name="count_during_typing",
    string='''
    =g>
        isa dual_goal
        count =num
        motor_busy yes
    ==>
    =g>
        motor_busy retrieving
    +retrieval>
        isa count_fact
        current =num
'''
)

dual_model.productionstring(
    name="update_count",
    string='''
    =g>
        isa dual_goal
        motor_busy retrieving
    =retrieval>
        isa count_fact
        next =next
    ==>
    =g>
        count =next
        motor_busy yes
    ~retrieval>
'''
)

print("Dual-task model created!")
print("Can count (cognitive) while typing (motor)")

## 5. Modeling Skilled vs Novice Typists

Motor parameters can be adjusted to model individual differences. An expert typist has faster motor execution and less preparation time compared to a novice.

In [ ]:
import pyactr as actr

expert_model = actr.ACTRModel()
novice_model = actr.ACTRModel()

expert_model.model_parameters["motor_burst_time"] = 0.05
expert_model.model_parameters["motor_initiation_time"] = 0.05

novice_model.model_parameters["motor_burst_time"] = 0.1
novice_model.model_parameters["motor_initiation_time"] = 0.15
novice_model.model_parameters["motor_feature_prep_time"] = 0.15

actr.chunktype("type_letter", "letter")

for model, name in [(expert_model, "Expert"), (novice_model, "Novice")]:
    model.productionstring(
        name="type",
        string='''
        =g>
            isa type_letter
            letter =key
        ?manual>
            state free
        ==>
        +manual>
            isa _manual
            cmd press_key
            key =key
        =g>
            letter done
    '''
    )
    
    prep = model.model_parameters["motor_initiation_time"]
    exec = model.model_parameters["motor_burst_time"]
    total = prep + exec
    wpm = 60 / (total * 5)
    
    print(f"{name} Typist:")
    print(f"  Key press time: {total:.3f}s")
    print(f"  Typing speed: ~{wpm:.0f} WPM")
    print()

## 6. Complete Interface Interaction Model

Bringing it all together: a model that finds a button in an interface, moves the mouse to it, and clicks. This combines visual search (retrieval of button location), motor planning, and execution.

In [ ]:
import pyactr as actr

interface_model = actr.ACTRModel()
interface_model.model_parameters["subsymbolic"] = True

actr.chunktype("button", "label x_pos y_pos width height")
actr.chunktype("task_goal", "state target_button mouse_x mouse_y")

buttons = [
    ("Submit", 100, 200, 80, 30),
    ("Cancel", 200, 200, 80, 30),
    ("Help", 300, 200, 60, 30)
]

for label, x, y, w, h in buttons:
    interface_model.decmem.add(
        actr.makechunk(
            chunktype="button",
            label=label,
            x_pos=str(x),
            y_pos=str(y),
            width=str(w),
            height=str(h)
        )
    )

interface_model.productionstring(
    name="find_button",
    string='''
    =g>
        isa task_goal
        state searching
        target_button =target
    ==>
    =g>
        state locating
    +retrieval>
        isa button
        label =target
'''
)

interface_model.productionstring(
    name="move_to_button",
    string='''
    =g>
        isa task_goal
        state locating
    =retrieval>
        isa button
        x_pos =x
        y_pos =y
    ?manual>
        state free
    ==>
    =g>
        state moving
        mouse_x =x
        mouse_y =y
    +manual>
        isa _manual
        cmd move_cursor
        screen_x =x
        screen_y =y
'''
)

interface_model.productionstring(
    name="click_button",
    string='''
    =g>
        isa task_goal
        state moving
    ?manual>
        state free
    ==>
    =g>
        state done
    +manual>
        isa _manual
        cmd click_mouse
'''
)

goal = actr.makechunk(
    chunktype="task_goal",
    state="searching",
    target_button="Submit"
)
interface_model.goal.add(goal)

print("Interface interaction model ready!")
print("Will: 1) Find Submit button, 2) Move mouse, 3) Click")

## Exercises

1. Modify the typing model to occasionally make mistakes based on finger distance between keys

2. Create a model that navigates through menu items using arrow keys

3. Model the motor actions for dragging an object from one location to another

4. Build two versions of a clicking model—one fast but error-prone, one slow but accurate

5. Compare models using mouse clicks vs touchscreen taps

## What's Next

Motor timing becomes especially important when modeling real-world tasks. A model that accurately predicts when someone will finish typing a password, or how long it takes to navigate a menu, can inform interface design. In the next tutorial, we'll look at how to combine all the ACT-R components we've covered to model more complex, realistic tasks.